## 0. Descarga de librerías

In [1]:
!pip install kagglehub sentence-transformers scikit-learn pandas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\david\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


O ejecutar el requirements.txt

## 1. Descarga del corpus

In [2]:
import kagglehub
partial_path = kagglehub.dataset_download("stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset")

c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Elección de datos

El corpus está dividido en dos archivos

- *rotten_tomatoes_critic_reviews*: Las críticas de cada película.
- *rotten_tomatoes_movies*: Información de las películas

Para tener un sistema de recuperación de información adecuado, se optó por unir las dos tablas, así guardamos la información de la película y las críticas de la misma.

In [3]:
import pandas as pd

movies_path = partial_path + '\\' + 'rotten_tomatoes_movies.csv'
movies = pd.read_csv(movies_path)
reviews_path = partial_path + '\\' + 'rotten_tomatoes_critic_reviews.csv'
reviews = pd.read_csv(reviews_path)

merged = reviews.merge(movies[['rotten_tomatoes_link', 'movie_title', 'genres', 'movie_info']], 
                        on='rotten_tomatoes_link', how='left')

display(merged)

,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_type,review_score,review_date,review_content,movie_title,genres,movie_info
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Always trouble-prone, the life of teenager Per..."
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff...",Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Always trouble-prone, the life of teenager Per..."
2,m/0814255,NaN,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Always trouble-prone, the life of teenager Per..."
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Always trouble-prone, the life of teenager Per..."
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Always trouble-prone, the life of teenager Per..."
...,...,...,...,...,...,...,...,...,...,...,...
1130012,m/zulu_dawn,Chuck O'Leary,False,Fantastica Daily,Rotten,2/5,2005-11-02,NaN,Zulu Dawn,"Action & Adventure, Art House & International,...",Sir Henry Bartle Frere's (John Mills) vastly o...
1130013,m/zulu_dawn,Ken Hanke,False,"Mountain Xpress (Asheville, NC)",Fresh,3.5/5,2007-03-07,"Seen today, it's not only a startling indictme...",Zulu Dawn,"Action & Adventure, Art House & International,...",Sir Henry Bartle Frere's (John Mills) vastly o...
1130014,m/zulu_dawn,Dennis Schwartz,False,Dennis Schwartz Movie Reviews,Fresh,B+,2010-09-16,A rousing visual spectacle that's a prequel of...,Zulu Dawn,"Action & Adventure, Art House & International,...",Sir Henry Bartle Frere's (John Mills) vastly o...
1130015,m/zulu_dawn,Christopher Lloyd,False,Sarasota Herald-Tribune,Rotten,3.5/5,2011-02-28,"A simple two-act story: Prelude to war, and th...",Zulu Dawn,"Action & Adventure, Art House & International,...",Sir Henry Bartle Frere's (John Mills) vastly o...


## 3. Preprocesamiento de texto.

Al utilizar emmbedings, se optó por un preprocesamiento básico del texto crudo.

In [4]:
import re

def preprocess_for_embeddings(raw_text: str) -> str:
    text = raw_text.lower()
    text = re.sub(r'[-/]', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

Se aplica el preprocesamiento a la información de la película y a las reviews. 

In [5]:
merged['clean_review'] = merged['review_content'].fillna('').apply(preprocess_for_embeddings)
merged['clean_info'] = merged['movie_info'].fillna('').apply(preprocess_for_embeddings)

## 4. Generación de emmbedings

Se unen las dos columnas preprocesadas para generar el texto que pasará el proceso de embedding.

In [6]:
merged['text_for_embedding'] = merged['clean_review'] + ' ' + merged['clean_info']

Por temas de poder de procesamiento y tiempo, se optó por agrupar las reseñas por pelicula, así no se generan emmbedings por reseña, sino por películas.

In [7]:
grouped = merged.groupby('rotten_tomatoes_link').agg({
    'movie_title': 'first',
    'genres': 'first',
    'movie_info': 'first',
    'text_for_embedding': ' '.join
}).reset_index()

Se carga el modelo

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5810.38it/s]


Se generan los emmbedings

In [9]:
document_embeddings = model.encode(grouped['text_for_embedding'].tolist(), show_progress_bar=False)

## 5. Procesamiento y generación de embeddings de los queries

In [10]:
queries = [
    "science fiction movie with advanced technology",
    "romantic story with emotional relationships",
    "action movie with intense fight scenes",
    "horror film that creates fear and suspense",
    "visually impressive movie with weak storyline",
    "emotionally moving performance by the lead actor",
    "predictable plot but entertaining experience",
    "movie praised by critics but unpopular with audiences"
]

clean_queries = [preprocess_for_embeddings(q) for q in queries]
query_embeddings = model.encode(clean_queries)

## 6. Similitud del coseno 

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

for i, query in enumerate(queries):
    similarities = cosine_similarity([query_embeddings[i]], document_embeddings)[0]
    
    grouped['similarity'] = similarities
    top_k = grouped.nlargest(10, 'similarity').copy()
    
    resultado = pd.DataFrame({
        'Ranking': range(1, 11),
        'ID Documento': top_k['rotten_tomatoes_link'].values,
        'Título Película': top_k['movie_title'].values,
        'Fragmento de Texto': top_k['movie_info'].fillna(top_k['text_for_embedding']).apply(lambda x: x[:200] + '...').values,
        'Similitud': top_k['similarity'].round(4).values
    })
    
    print(f"\nQ{i+1}: {query}")
    display(resultado)


Q1: science fiction movie with advanced technology


,Ranking,ID Documento,Título Película,Fragmento de Texto,Similitud
0,1,m/kill_command,Kill Command,"In the near future, a training exercise leads ...",0.4992
1,2,m/2050,2050,2050 is ostensibly a film about sex robots han...,0.4745
2,3,m/joker_3d,Joker 3D,A NASA scientist who's trying to create a devi...,0.4626
3,4,m/real_genius,Real Genius,When science whiz Mitch Taylor (Gabe Jarret) a...,0.4530
4,5,m/tomorrowland_2015,Tomorrowland,Whenever Casey Newton (Britt Robertson) touche...,0.4528
5,6,m/the_machine_2013,The Machine,Two scientists fall in love while they are cre...,0.4520
6,7,m/ai_artificial_intelligence,A.I. Artificial Intelligence,"A robotic boy, the first programmed to love, D...",0.4395
7,8,m/proximity_2020,Proximity,if youre a hardcore sci fi devotee then i reco...,0.4343
8,9,m/spymate_2006,Spymate,A former agent (Chris Potter) and his onetime ...,0.4308
9,10,m/terminator_2_judgment_day,Terminator 2: Judgment Day,"In this sequel set eleven years after ""The Ter...",0.4306



Q2: romantic story with emotional relationships


,Ranking,ID Documento,Título Película,Fragmento de Texto,Similitud
0,1,m/1159365-standing_still,Standing Still,A chain reaction of confrontations and romanti...,0.5363
1,2,m/almost_love,Almost Love,A couple are about to find out if their relati...,0.5039
2,3,m/joy_2014,Joy,"A story of a family across four generations, c...",0.4834
3,4,m/her,Her,A sensitive and soulful man earns a living by ...,0.4828
4,5,m/a_promise_2013,A Promise,"In the early 20th century, a love triangle for...",0.4818
5,6,m/soulmate_2016,SoulMate,A woman's life is disrupted when a book is wri...,0.4718
6,7,m/felix_and_meira,Felix & Meira,A young married woman finds freedom from the O...,0.4713
7,8,m/la_mujer_de_mi_hermano,La mujer de mi hermano,An attractive young woman's search for passion...,0.4597
8,9,m/all_the_bright_places,All the Bright Places,"After meeting each other, two people struggle ...",0.4538
9,10,m/love_story,Love Story,When wealthy Harvard University law student Ol...,0.4513



Q3: action movie with intense fight scenes


...

,Ranking,ID Documento,Título Película,Fragmento de Texto,Similitud
0,1,m/beginners,Beginners,"After his mother dies, Oliver (Ewan McGregor) ...",0.4684
1,2,m/casting_about,Casting About,Filmmaker Barry J. Hershey documents the proce...,0.4641
2,3,m/face_to_face_2011,Face to Face,Tempers and emotions flare when 10 people have...,0.4556
3,4,m/i_am_jonas,I Am Jonas (Jonas),Two moments of a man's life intertwine....,0.4499
4,5,m/casting_by,Casting By,"Interviews with Martin Scorsese, Woody Allen, ...",0.4489
5,6,m/le_samourai,Le samouraï,Hit man Jef Costello (Alain Delon) goes throug...,0.4462
6,7,m/beautiful_mind,A Beautiful Mind,A human drama inspired by events in the life o...,0.4383
7,8,m/felt,Felt,A woman (Amy Everson) creates costumed alter e...,0.4369
8,9,m/glengarry_glen_ross,Glengarry Glen Ross,When an office full of New York City real esta...,0.4338
9,10,m/newly_single,Newly Single,The life of a man on the verge of success is d...,0.4331



Q7: predictable plot but entertaining experience


,Ranking,ID Documento,Título Película,Fragmento de Texto,Similitud
0,1,m/cheap_thrills_2013,Cheap Thrills,A series of escalating bets pits recently reun...,0.5396
1,2,m/the_fare,The Fare,the performances in what is mostly a two hande...,0.5332
2,3,m/what_we_found,What We Found,the quality of the film seems to replicate tha...,0.5281
3,4,m/30_miles_from_nowhere,30 Miles from Nowhere,its offbeat witty and completely entertaining ...,0.5117
4,5,m/set_it_up,Set It Up,Two overworked and underpaid assistants come u...,0.5098
5,6,m/strange_but_true,Strange But True,A woman tells her deceased boyfriend's family ...,0.5030
6,7,m/the_good_neighbor_2016,The Good Neighbor,A pair of mischievous high school kids (Logan ...,0.4997
7,8,m/tone_deaf,Tone-Deaf,After a string of bad relationships and work f...,0.4990
8,9,m/i_inside,The I Inside,"After a near-death experience, a man (Ryan Phi...",0.4954
9,10,m/hospitality,Hospitality,A former prostitute must protect her son and h...,0.4887



Q8: movie praised by critics but unpopular with audiences


,Ranking,ID Documento,Título Película,Fragmento de Texto,Similitud
0,1,m/red_hollywood,Red Hollywood,makes a significant and entertaining contribut...,0.5294
1,2,m/left_behind_the_movie,Left Behind: The Movie,a little hard to believe despite minor flaws ...,0.5080
2,3,m/1110646-uprising,Uprising,a solid network effort about heroism in a most...,0.5021
3,4,m/10010957-polanski_unauthorized,Polanski (Polanski Unauthorized),Filmmaker Roman Polanski (Damian Chapa) courts...,0.4733
4,5,m/1159169-for_your_consideration,For Your Consideration,The possibility of Oscar gold holds the cast a...,0.4730
5,6,m/nothing_but_the_blood,Nothing But the Blood,"A controversial cult moves into a small town, ...",0.4669
6,7,m/attack-of-the-killer-tomatoes,Attack of the Killer Tomatoes!,"In a spoof of low-budget sci-fi films, oozing ...",0.4665
7,8,m/lost_in_america_2020,Lost in America,Chronicles homelessness in the United States f...,0.4620
8,9,m/playback_2011,Playback,"While digging into their town's dark past, hig...",0.4619
9,10,m/still_here,Still Here,while some aspects of this movie feel off the ...,0.4539


## 7. Tabla resumen

In [12]:
resumen = []

for i, query in enumerate(queries):
    similarities = cosine_similarity([query_embeddings[i]], document_embeddings)[0]
    grouped['similarity'] = similarities
    top_1 = grouped.nlargest(1, 'similarity').iloc[0]
    
    resumen.append({
        'Consulta': f"Q{i+1}: {query}",
        'Documento Top-1': top_1['rotten_tomatoes_link'],
        'Título Película': top_1['movie_title'],
        'Similitud': round(top_1['similarity'], 4)
    })

tabla_resumen = pd.DataFrame(resumen)
display(tabla_resumen)

,Consulta,Documento Top-1,Título Película,Similitud
0,Q1: science fiction movie with advanced techno...,m/kill_command,Kill Command,0.4992
1,Q2: romantic story with emotional relationships,m/1159365-standing_still,Standing Still,0.5363
2,Q3: action movie with intense fight scenes,m/a_violent_man,A Violent Man,0.5461
3,Q4: horror film that creates fear and suspense,m/wildling,Wildling,0.6586
4,Q5: visually impressive movie with weak storyline,m/what_we_found,What We Found,0.5582
5,Q6: emotionally moving performance by the lead...,m/beginners,Beginners,0.4684
6,Q7: predictable plot but entertaining experience,m/cheap_thrills_2013,Cheap Thrills,0.5396
7,Q8: movie praised by critics but unpopular wit...,m/red_hollywood,Red Hollywood,0.5294


## 6. Desafío de excelencia

### Cargar modelo

In [13]:
model_2 = SentenceTransformer('paraphrase-MiniLM-L3-v2')

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1428.79it/s]


### Calculo de embeddings para el corpus (agrupado) y queries

In [14]:
document_embeddings_2 = model_2.encode(grouped['text_for_embedding'].tolist(), show_progress_bar=False)
query_embeddings_2 = model_2.encode(clean_queries)

### Calculo de similitudes con los nuevos embeddings

In [15]:
## Diccionario para comparar resultados
comparacion = []

for i, query in enumerate(queries):
    sim_2 = cosine_similarity([query_embeddings_2[i]], document_embeddings_2)[0]
    grouped['sim_model_2'] = sim_2
    top1_m2 = grouped.nlargest(1, 'sim_model_2').iloc[0]
    
    comparacion.append({
        'Consulta': query,
        'Top-1 MiniLM': resumen[i]['Título Película'],
        'Similitud MiniLM': resumen[i]['Similitud'],
        'Top-1 MiniLM-L3-v2': top1_m2['movie_title'],
        'Similitud MiniLM-L3-v2': round(top1_m2['sim_model_2'], 4),
        'Coinciden': 'Sí' if resumen[i]['Documento Top-1'] == top1_m2['rotten_tomatoes_link'] else 'No'
    })

tabla_comparacion = pd.DataFrame(comparacion)
display(tabla_comparacion)

,Consulta,Top-1 MiniLM,Similitud MiniLM,Top-1 MiniLM-L3-v2,Similitud MiniLM-L3-v2,Coinciden
0,science fiction movie with advanced technology,Kill Command,0.4992,Minority Report,0.5316,No
1,romantic story with emotional relationships,Standing Still,0.5363,Jackie & Ryan,0.6076,No
2,action movie with intense fight scenes,A Violent Man,0.5461,Beowulf,0.5752,No
3,horror film that creates fear and suspense,Wildling,0.6586,Impossible Monsters,0.6881,No
4,visually impressive movie with weak storyline,What We Found,0.5582,Left Behind: The Movie,0.5667,No
5,emotionally moving performance by the lead actor,Beginners,0.4684,6 Month Rule,0.5020,No
6,predictable plot but entertaining experience,Cheap Thrills,0.5396,Scarborough,0.5057,No
7,movie praised by critics but unpopular with au...,Red Hollywood,0.5294,Blame,0.5437,No


In [22]:
query = 'Science fiction movie with advanced technology'
query_embeddings_2 = model_2.encode(query)

sim_2 = cosine_similarity([query_embeddings_2], document_embeddings_2)[0]
grouped['sim_model_2'] = sim_2
top_k = grouped.nlargest(5, 'sim_model_2').copy()

print(top_k['movie_title'].values)


['Minority Report' 'Planet of the Apes' 'Terminator 2: Judgment Day'
 'A.I. Artificial Intelligence' 'Southland Tales']


In [24]:
prompt = f'Eres un chatbot conversacional. Mi usuario quiere saber acerca de {query}. Mi sistema de recuperación de información me ha dado los siguientes documentos relevantes: {top_k["movie_title"].values}. Por favor, responde a la consulta del usuario utilizando esta información de manera coherente y completa.'

In [25]:
import os
from google import genai

# Configura tu API Key
client = genai.Client(api_key=api_key)

# Realiza la consulta utilizando el modelo estándar de Gemini
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

print(response.text)

Claro, hay varias películas de ciencia ficción en tu lista que presentan tecnología avanzada como elemento central o significativo en sus tramas. Aquí te detallo cómo cada una aborda este aspecto:

1.  **Minority Report**: Esta película es un excelente ejemplo de un futuro cercano impulsado por la tecnología. Presenta sistemas de "precrimen" que utilizan la precognición para predecir y prevenir asesinatos antes de que ocurran, lo que implica una tecnología de monitoreo y análisis extremadamente sofisticada. También se ven interfaces de usuario basadas en gestos (donde el protagonista manipula pantallas holográficas con movimientos de sus manos), vehículos autoconducidos de levitación magnética (maglev), publicidad personalizada basada en escaneos de retina y sistemas de identificación personal a gran escala.

2.  **Terminator 2: Judgment Day**: La tecnología avanzada es el corazón de esta saga. La película se centra en la inteligencia artificial (IA) avanzada con Skynet, una red de def

In [26]:
from collections import deque

contexto = deque()
contexto.append(prompt)
contexto.append(response.text)
contexto.append('Ya pero cuál de estas es más antigua?')

In [28]:
print(str(contexto))

deque(["Eres un chatbot conversacional. Mi usuario quiere saber acerca de Science fiction movie with advanced technology. Mi sistema de recuperación de información me ha dado los siguientes documentos relevantes: ['Minority Report' 'Planet of the Apes' 'Terminator 2: Judgment Day'\n 'A.I. Artificial Intelligence' 'Southland Tales']. Por favor, responde a la consulta del usuario utilizando esta información de manera coherente y completa.", 'Claro, hay varias películas de ciencia ficción en tu lista que presentan tecnología avanzada como elemento central o significativo en sus tramas. Aquí te detallo cómo cada una aborda este aspecto:\n\n1.  **Minority Report**: Esta película es un excelente ejemplo de un futuro cercano impulsado por la tecnología. Presenta sistemas de "precrimen" que utilizan la precognición para predecir y prevenir asesinatos antes de que ocurran, lo que implica una tecnología de monitoreo y análisis extremadamente sofisticada. También se ven interfaces de usuario basa

In [29]:
# Realiza la consulta utilizando el modelo estándar de Gemini
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=contexto,
)

print(response.text)

De las películas que mencionaste, **Planet of the Apes** (El planeta de los simios) es la más antigua, con la película original estrenada en **1968**.
